# Discrete Latent Geometry Demo

Lightweight inspection notebook for CLI-generated latent-geometry diagnostics. The default preset is the promoted standard VQ tokenizer; RVQ q2 is available only as an ablation preset.


## Parameters

Switch `PRESET` between `standard_vq` and `rvq_q2`. Leave `RUN_ANALYSIS=False` when inspecting existing generated summaries and plots.


In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import Image, Markdown, display

PRESET = "standard_vq"  # choices: "standard_vq", "rvq_q2"
RUN_ANALYSIS = False
BASE_DATA_DIR = Path("data/processed")

PRESETS = {
    "standard_vq": {
        "label": "Standard VQ promoted baseline",
        "config": Path("configs/experiments/sp500_vix_causal_vq_tokenizer.yaml"),
        "tokenizer_dir": Path("outputs/sp500_vix_discrete/tokenizer/sp500_vix_causal_vq_tokenizer_seed0"),
        "token_data_dir": Path("outputs/sp500_vix_discrete/token_prior/tokens_codebook64_codebookdim16"),
        "output_dir": Path("outputs/latent_geometry/sp500_vix_standard_vq"),
    },
    "rvq_q2": {
        "label": "RVQ q2 ablation",
        "config": Path("configs/experiments/sp500_vix_causal_rvq_tokenizer_q2.yaml"),
        "tokenizer_dir": Path("outputs/sp500_vix_discrete/vq_family_tokenizer_ablation/sp500_vix_causal_rvq_tokenizer_q2_seed0"),
        "token_data_dir": Path("outputs/sp500_vix_discrete/token_prior/tokens_rvq_q2"),
        "output_dir": Path("outputs/latent_geometry/sp500_vix_rvq_q2"),
    },
}

if PRESET not in PRESETS:
    raise ValueError(f"Unknown PRESET: {PRESET}")

SELECTED = PRESETS[PRESET]
CONFIG_PATH = SELECTED["config"]
TOKENIZER_DIR = SELECTED["tokenizer_dir"]
TOKEN_DATA_DIR = SELECTED["token_data_dir"]
OUTPUT_DIR = SELECTED["output_dir"]

print(SELECTED["label"])
print("output_dir:", OUTPUT_DIR)


## Artefact Check

Missing paths indicate that the CLI training or token-extraction step should be run before analysis. The notebook does not point to private milestone directories.


In [ ]:
required_inputs = {
    "CONFIG_PATH": CONFIG_PATH,
    "TOKENIZER_DIR": TOKENIZER_DIR,
    "TOKEN_DATA_DIR": TOKEN_DATA_DIR,
    "BASE_DATA_DIR": BASE_DATA_DIR,
}
missing = {name: path for name, path in required_inputs.items() if not path.exists()}

if missing:
    print("Missing required artefacts for a real latent-geometry run:")
    for name, path in missing.items():
        print(f"  - {name}: {path}")
    if PRESET == "standard_vq":
        print("\nCreate standard VQ artefacts from the repository root:")
        print("poetry run tcvae-train-tokenizer --config configs/experiments/sp500_vix_causal_vq_tokenizer.yaml --output-dir outputs/sp500_vix_discrete/tokenizer --base-data-dir data/processed --no-wandb")
        print("poetry run python scripts/extract_token_indices.py --config configs/experiments/sp500_vix_causal_vq_tokenizer.yaml --tokenizer-dir <tokenizer-dir> --output-dir outputs/sp500_vix_discrete/token_prior/tokens_codebook64_codebookdim16 --base-data-dir data/processed --seed 99")
else:
    print("All configured inputs are present.")
    for name, path in required_inputs.items():
        print(f"  - {name}: {path}")


## Optional Analysis Run

This invokes the same script used for the verification notes. W&B is intentionally not enabled from the notebook.


In [ ]:
analysis_command = [
    sys.executable,
    "scripts/analyze_discrete_latent_geometry.py",
    "--config",
    str(CONFIG_PATH),
    "--tokenizer-dir",
    str(TOKENIZER_DIR),
    "--token-data-dir",
    str(TOKEN_DATA_DIR),
    "--output-dir",
    str(OUTPUT_DIR),
    "--base-data-dir",
    str(BASE_DATA_DIR),
    "--plot-voronoi",
]

if RUN_ANALYSIS:
    if missing:
        print("RUN_ANALYSIS=True, but required inputs are missing. Resolve the paths above first.")
    else:
        print("Running:")
        print(" ".join(analysis_command))
        subprocess.run(analysis_command, check=True)
else:
    print("RUN_ANALYSIS=False; using any existing files under", OUTPUT_DIR)


## Numeric Summary

JSON and Markdown summaries are the decision inputs. PNGs are for inspection and report figure selection.


In [ ]:
summary_path = OUTPUT_DIR / "codebook_geometry_summary.json"
markdown_path = OUTPUT_DIR / "latent_geometry_summary.md"
pair_path = OUTPUT_DIR / "q0_q1_pair_summary.json"

if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    usage = summary.get("usage", {})
    geometry = summary.get("geometry", {})
    metadata = summary.get("metadata", {})
    display(pd.DataFrame([
        {
            "quantizer_type": metadata.get("quantizer_type"),
            "embedding_shape": geometry.get("embedding_shape"),
            "projection_method": geometry.get("projection_method"),
            "active_code_count": usage.get("active_code_count"),
            "perplexity": usage.get("codebook_perplexity"),
            "entropy": usage.get("entropy"),
        }
    ]))
    if usage.get("per_quantizer"):
        display(Markdown("### Per-quantizer usage"))
        display(pd.DataFrame(usage["per_quantizer"]))
    if summary.get("condition_buckets"):
        display(Markdown("### VIX-bucket usage"))
        display(pd.DataFrame(summary["condition_buckets"]))
else:
    print(f"No JSON summary found at {summary_path}")

if pair_path.exists():
    pair_summary = json.loads(pair_path.read_text())
    fields = ["pair_count", "active_pair_count", "active_pair_ratio", "absent_pair_mass", "zero_count_pairs"]
    display(Markdown("### q0/q1 pair support"))
    display(pd.DataFrame([{field: pair_summary.get(field) for field in fields}]))

if markdown_path.exists():
    print("Markdown summary excerpt:")
    print(markdown_path.read_text()[:1600])
else:
    print(f"No Markdown summary found at {markdown_path}")


## Generated Plots

Displayed images are not embedded after output stripping. For the promoted report, prefer the standard VQ projection, usage projection, VIX-bucket usage, and token trajectory plots.


In [ ]:
plot_names = [
    "codebook_projection.png",
    "codebook_usage_projection.png",
    "vix_bucket_code_usage.png",
    "token_trajectory_examples.png",
    "codebook_voronoi.png",
    "codebook_nearest_region.png",
    "q0_q1_pair_heatmap.png",
]

rows = []
for plot_name in plot_names:
    plot_path = OUTPUT_DIR / plot_name
    rows.append({"plot": plot_name, "path": str(plot_path), "exists": plot_path.exists()})
display(pd.DataFrame(rows))

for plot_name in plot_names:
    plot_path = OUTPUT_DIR / plot_name
    if plot_path.exists():
        display(Markdown(f"### `{plot_name}`"))
        display(Image(filename=str(plot_path)))


## Interpretation

Standard VQ remains the promoted public baseline: it has broad code utilisation, VIX-sensitive usage structure, and a simple one-code interface for the additive scalar-conditioned causal AR prior.

RVQ q2 remains an ablation. Its coarse/detail geometry is informative, but the sparse same-time q0/q1 support makes generation harder. GroupedRVQ and MGVQ should remain future work unless a measured standard-VQ failure mode justifies them.
